In [18]:
import pandas as pd, numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.model_selection import  RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')


In [19]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')   

In [20]:
x_train = train_df.drop(columns=['Activity'])
y_train = train_df['Activity']
x_test = test_df.drop(columns=['Activity'])
y_test = test_df['Activity']

In [21]:
print('Calculating corealtion matrix')
corr_matrix =  x_train.corr().abs()

print('selecting upper triangle')
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))

print('removing highly corelated columns ')
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]

final_x_train = x_train.drop(columns=to_drop)

final_x_test = x_test.drop(columns=to_drop)

Calculating corealtion matrix
selecting upper triangle
removing highly corelated columns 


In [22]:
y_train.value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('scaler', StandardScaler(), final_x_train.columns)
])

final_y_train = y_train.map({'LAYING':0, 'STANDING':1, 'SITTING':2, 'WALKING':3, 'WALKING_UPSTAIRS':4, 'WALKING_DOWNSTAIRS':5})
final_y_test = y_test.map({'LAYING':0, 'STANDING':1, 'SITTING':2, 'WALKING':3, 'WALKING_UPSTAIRS':4, 'WALKING_DOWNSTAIRS':5})


In [24]:
models = {
    'Random Forest': RandomForestClassifier(),
    'Extra Trees': ExtraTreesClassifier(),
}
parameters = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [ 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    },
    'Extra Trees': {
        'n_estimators': [100, 200, 300],
        'max_depth': [ 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'bootstrap': [True, False]
    }
}

In [27]:
best_models = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    my_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    current_parameters = {}
    for param_name, param_values in parameters[model_name].items():
        current_parameters[f'classifier__{param_name}'] = param_values
    random_search = RandomizedSearchCV(my_pipeline,
                                        current_parameters,
                                        n_iter=10,
                                        cv=5, 
                                        scoring='f1_macro',
                                        n_jobs=-1,
                                        random_state=42)
    random_search.fit(final_x_train, final_y_train)
    best_models[model_name] = random_search.best_estimator_
    print(f"Optimal Parameters for {model_name}: {random_search.best_params_}")
    print(f"Best F1 Score for {model_name}: {random_search.best_score_}")

Training Random Forest...
Optimal Parameters for Random Forest: {'classifier__n_estimators': 100, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 4, 'classifier__max_depth': 20, 'classifier__bootstrap': True}
Best F1 Score for Random Forest: 0.9216626152168315
Training Extra Trees...
Optimal Parameters for Extra Trees: {'classifier__n_estimators': 300, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 1, 'classifier__max_depth': 20, 'classifier__bootstrap': False}
Best F1 Score for Extra Trees: 0.9250915800204821


In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report



print("\n--- MODELS EVALUATION STARTED ---")
for name, model in best_models.items():
    print(f"\nEvaluating: {name}...")
    
    y_pred = model.predict(final_x_test)
    
    accuracy = accuracy_score(final_y_test, y_pred)
    f1 = f1_score(final_y_test, y_pred, average='macro')
    classification_rep = classification_report(final_y_test, y_pred)

    print(f"Accuracy: {accuracy}")
    print(f"F1 Score: {f1}")
    print("Classification Report:\n", classification_rep)
    print('done')


--- MODELS EVALUATION STARTED ---

Evaluating: Random Forest...
Accuracy: 0.9317950458092976
F1 Score: 0.9309618884990436
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       537
           1       0.89      0.89      0.89       532
           2       0.89      0.89      0.89       491
           3       0.93      0.99      0.96       496
           4       0.91      0.91      0.91       471
           5       0.97      0.90      0.93       420

    accuracy                           0.93      2947
   macro avg       0.93      0.93      0.93      2947
weighted avg       0.93      0.93      0.93      2947

done

Evaluating: Extra Trees...
Accuracy: 0.9429928741092637
F1 Score: 0.9420098691151022
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       537
           1       0.89      0.95      0.92       532
           2       0.94      0.8